# Big Data Bottleneck Analysis — `scan_events`

Analisis PySpark untuk menghitung **dwell time** setiap kombinasi `package_id × hub_id` dari pasangan scan `ARRIVAL → DEPARTURE`, memeriksa kualitas data, dan menentukan hub yang perlu diprioritaskan untuk investigasi operasional.

Alur utama: **Filter → Group → Aggregate → Sort**.

## 0. Konsep Big Data: 5 Vs dan Keterbatasan Pandas

- **Volume**: jumlah event dapat tumbuh menjadi jutaan atau miliaran baris sehingga tidak selalu muat di memori satu komputer.
- **Velocity**: scan ARRIVAL dan DEPARTURE terus masuk dengan cepat dan perlu diproses tepat waktu.
- **Variety**: data dapat berasal dari scanner, aplikasi kurir, API, atau sistem hub dengan format yang berbeda.
- **Veracity**: missing scan, duplicate event, timestamp rusak, dan urutan event yang salah dapat menyesatkan analisis.
- **Value**: data scan baru bernilai ketika dapat menunjukkan bottleneck dan mengarahkan tindakan operasional.

Pandas cocok untuk data kecil hingga menengah pada satu mesin, tetapi pemrosesannya dibatasi RAM dan CPU mesin tersebut. PySpark membagi data dan komputasi ke beberapa partition/worker, sehingga operasi filter, group, aggregate, dan sort dapat diskalakan. Pandas hanya digunakan di bagian akhir untuk mengekspor hasil agregasi yang ukurannya kecil ke CSV.

## 1. Setup PySpark

In [30]:
from pathlib import Path
from datetime import datetime, timedelta
import random

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("Anteraja-BigData-BottleneckAnalysis")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

NOTEBOOK_CWD = Path.cwd()
PROJECT_ROOT = NOTEBOOK_CWD.parent if NOTEBOOK_CWD.name == "analysis" else NOTEBOOK_CWD
OUTPUT_DIR = PROJECT_ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Spark version:", spark.version)
print("Project root:", PROJECT_ROOT)
print("Output directory:", OUTPUT_DIR)

Spark version: 4.2.0
Project root: /home/rian/Documents/anteraja-capstone-rianyuliawan/bigdata-bottleneck-analysis
Output directory: /home/rian/Documents/anteraja-capstone-rianyuliawan/bigdata-bottleneck-analysis/output


## 2. Dataset dan Schema

Dataset simulasi menggunakan schema mentah:

`package_id STRING | hub_id STRING | event_type STRING | timestamp STRING`

Simulasi sengaja menyisipkan missing ARRIVAL/DEPARTURE, duplicate scan, timestamp tidak valid, dan DEPARTURE sebelum ARRIVAL agar pemeriksaan kualitas dapat diverifikasi.

In [31]:
SEED = 42
N_PACKAGES = 2500

HUB_WEIGHTS = {
    "HUB_JKT": 0.35,
    "HUB_MKS": 0.25,
    "HUB_SBY": 0.15,
    "HUB_BDG": 0.10,
    "HUB_DPS": 0.08,
    "HUB_YGY": 0.04,
    "HUB_BKS": 0.03,
}

# Rata-rata simulasi dibuat berbeda agar bottleneck dapat dianalisis secara bermakna.
HUB_DWELL_MEAN = {
    "HUB_JKT": 7.0,
    "HUB_MKS": 10.5,
    "HUB_SBY": 8.5,
    "HUB_BDG": 7.5,
    "HUB_DPS": 6.5,
    "HUB_YGY": 9.0,
    "HUB_BKS": 6.0,
}

rng = random.Random(SEED)
hubs = list(HUB_WEIGHTS)
weights = list(HUB_WEIGHTS.values())
rows = []

for package_number in range(1, N_PACKAGES + 1):
    package_id = f"PKG_{package_number:06d}"
    hub_id = rng.choices(hubs, weights=weights, k=1)[0]
    arrival = datetime(2026, 10, 1) + timedelta(minutes=rng.randint(0, 15 * 24 * 60))
    dwell_hours = max(0.25, rng.gauss(HUB_DWELL_MEAN[hub_id], 2.5))
    departure = arrival + timedelta(hours=dwell_hours)

    missing_arrival = rng.random() < 0.015
    missing_departure = rng.random() < 0.08
    if missing_arrival and missing_departure:
        missing_departure = False

    invalid_order = rng.random() < 0.012 and not missing_arrival and not missing_departure
    if invalid_order:
        departure = arrival - timedelta(hours=rng.uniform(0.25, 3.0))

    arrival_timestamp = arrival.isoformat()
    departure_timestamp = departure.isoformat()

    if not missing_arrival:
        rows.append((package_id, hub_id, "ARRIVAL", arrival_timestamp))
    if not missing_departure:
        rows.append((package_id, hub_id, "DEPARTURE", departure_timestamp))

    # Duplicate persis.
    if not missing_arrival and rng.random() < 0.01:
        rows.append((package_id, hub_id, "ARRIVAL", arrival_timestamp))

    # Duplicate logis: event sama dengan timestamp berbeda.
    if not missing_arrival and rng.random() < 0.02:
        duplicate_arrival = arrival + timedelta(minutes=rng.randint(1, 20))
        rows.append((package_id, hub_id, "ARRIVAL", duplicate_arrival.isoformat()))

    # Timestamp rusak pada event tambahan agar event utama tetap dapat dipasangkan.
    if rng.random() < 0.01:
        rows.append((package_id, hub_id, rng.choice(["ARRIVAL", "DEPARTURE"]), "INVALID_TIMESTAMP"))

schema = "package_id STRING, hub_id STRING, event_type STRING, timestamp STRING"
df_raw = spark.createDataFrame(rows, schema=schema).cache()

print(f"Jumlah paket simulasi: {N_PACKAGES:,}")
print(f"Jumlah event mentah: {df_raw.count():,}")
df_raw.printSchema()
df_raw.show(5, truncate=False)

Jumlah paket simulasi: 2,500
Jumlah event mentah: 4,865
root
 |-- package_id: string (nullable = true)
 |-- hub_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- timestamp: string (nullable = true)

+----------+-------+----------+--------------------------+
|package_id|hub_id |event_type|timestamp                 |
+----------+-------+----------+--------------------------+
|PKG_000001|HUB_SBY|ARRIVAL   |2026-10-01T13:39:00       |
|PKG_000001|HUB_SBY|DEPARTURE |2026-10-01T22:03:02.038903|
|PKG_000002|HUB_JKT|ARRIVAL   |2026-10-06T07:03:00       |
|PKG_000003|HUB_MKS|ARRIVAL   |2026-10-01T03:32:00       |
|PKG_000003|HUB_MKS|DEPARTURE |2026-10-01T14:06:53.610364|
+----------+-------+----------+--------------------------+
only showing top 5 rows


## 3. Data Quality Check

Pemeriksaan dilakukan sebelum menghitung dwell time. Missing event dihitung pada tingkat `package_id × hub_id`, sedangkan duplicate scan dibedakan menjadi duplicate persis dan kelebihan event dalam kelompok paket–hub–event type.

In [32]:
# Parser toleran: timestamp rusak menjadi NULL, bukan menghentikan notebook.
df_parsed = df_raw.withColumn("ts", F.try_to_timestamp("timestamp")).cache()

total_events = df_raw.count()
total_package_hubs = df_raw.select("package_id", "hub_id").distinct().count()
null_counts = {
    column: df_raw.filter(F.col(column).isNull()).count()
    for column in df_raw.columns
}
invalid_timestamp_count = df_parsed.filter(F.col("ts").isNull()).count()

event_presence = (
    df_raw.groupBy("package_id", "hub_id")
    .agg(
        F.max(F.when(F.col("event_type") == "ARRIVAL", 1).otherwise(0)).alias("has_arrival"),
        F.max(F.when(F.col("event_type") == "DEPARTURE", 1).otherwise(0)).alias("has_departure"),
    )
)
missing_arrival_count = event_presence.filter(F.col("has_arrival") == 0).count()
missing_departure_count = event_presence.filter(F.col("has_departure") == 0).count()

exact_duplicate_count = total_events - df_raw.dropDuplicates().count()

event_group_counts = df_raw.groupBy("package_id", "hub_id", "event_type").count()
duplicate_event_groups = event_group_counts.filter(F.col("count") > 1).count()
duplicate_extra_scans = (
    event_group_counts
    .select(F.sum(F.greatest(F.col("count") - 1, F.lit(0))).alias("extra"))
    .first()["extra"]
)

print("=== DATA QUALITY — RAW EVENTS ===")
print("Total events:", total_events)
print("Total package × hub:", total_package_hubs)
print("Null per kolom:", null_counts)
print("Missing ARRIVAL:", missing_arrival_count)
print("Missing DEPARTURE:", missing_departure_count)
print("Invalid timestamp:", invalid_timestamp_count)
print("Exact duplicate rows:", exact_duplicate_count)
print("Duplicate event groups:", duplicate_event_groups)
print("Duplicate extra scans:", duplicate_extra_scans)

=== DATA QUALITY — RAW EVENTS ===
Total events: 4865
Total package × hub: 2500
Null per kolom: {'package_id': 0, 'hub_id': 0, 'event_type': 0, 'timestamp': 0}
Missing ARRIVAL: 44
Missing DEPARTURE: 201
Invalid timestamp: 25
Exact duplicate rows: 32
Duplicate event groups: 109
Duplicate extra scans: 110


## 4. Membentuk Pasangan Scan Valid

Langkah cleaning:

1. Gunakan hanya event ARRIVAL dan DEPARTURE dengan timestamp valid.
2. Hapus duplicate persis.
3. Ambil scan valid pertama untuk setiap `package_id × hub_id × event_type`.
4. Pasangkan ARRIVAL dan DEPARTURE berdasarkan `package_id × hub_id`.
5. Pisahkan pasangan dengan `DEPARTURE <= ARRIVAL` sebagai invalid order.
6. Hitung dwell time hanya dari pasangan valid.

In [33]:
valid_event_types = ["ARRIVAL", "DEPARTURE"]

df_usable = (
    df_parsed
    .filter(F.col("event_type").isin(valid_event_types))
    .filter(F.col("ts").isNotNull())
    .dropDuplicates(["package_id", "hub_id", "event_type", "timestamp"])
)

first_event_window = (
    Window.partitionBy("package_id", "hub_id", "event_type")
    .orderBy(F.col("ts").asc())
)

df_first_events = (
    df_usable
    .withColumn("row_number", F.row_number().over(first_event_window))
    .filter(F.col("row_number") == 1)
    .drop("row_number")
)

arrivals = (
    df_first_events
    .filter(F.col("event_type") == "ARRIVAL")
    .select("package_id", "hub_id", F.col("ts").alias("arrival_ts"))
)

departures = (
    df_first_events
    .filter(F.col("event_type") == "DEPARTURE")
    .select("package_id", "hub_id", F.col("ts").alias("departure_ts"))
)

complete_pairs = arrivals.join(departures, ["package_id", "hub_id"], "inner").cache()
invalid_order_count = complete_pairs.filter(F.col("departure_ts") <= F.col("arrival_ts")).count()

valid_dwell = (
    complete_pairs
    .filter(F.col("departure_ts") > F.col("arrival_ts"))
    .withColumn(
        "dwell_hours",
        (F.col("departure_ts").cast("long") - F.col("arrival_ts").cast("long")) / F.lit(3600.0),
    )
    .cache()
)

valid_pair_count = valid_dwell.count()
valid_pair_coverage_pct = valid_pair_count / total_package_hubs * 100

print("Complete ARRIVAL–DEPARTURE pairs:", complete_pairs.count())
print("DEPARTURE sebelum/sama dengan ARRIVAL:", invalid_order_count)
print("Valid pairs:", valid_pair_count)
print(f"Valid pair coverage: {valid_pair_coverage_pct:.2f}%")
valid_dwell.show(5, truncate=False)

Complete ARRIVAL–DEPARTURE pairs: 2254
DEPARTURE sebelum/sama dengan ARRIVAL: 18
Valid pairs: 2236
Valid pair coverage: 89.44%
+----------+-------+-------------------+--------------------------+------------------+
|package_id|hub_id |arrival_ts         |departure_ts              |dwell_hours       |
+----------+-------+-------------------+--------------------------+------------------+
|PKG_000166|HUB_SBY|2026-10-15 09:00:00|2026-10-15 15:15:42.591202|6.261666666666667 |
|PKG_000439|HUB_JKT|2026-10-14 07:02:00|2026-10-14 15:53:40.718743|8.86111111111111  |
|PKG_000585|HUB_MKS|2026-10-03 07:54:00|2026-10-03 19:01:32.308511|11.125555555555556|
|PKG_001358|HUB_MKS|2026-10-04 02:20:00|2026-10-04 11:15:48.520384|8.93              |
|PKG_001424|HUB_DPS|2026-10-15 07:07:00|2026-10-15 14:54:13.68712 |7.786944444444444 |
+----------+-------+-------------------+--------------------------+------------------+
only showing top 5 rows


## 5. Kualitas Pasangan Scan per Hub

In [34]:
hub_base = (
    df_raw.select("package_id", "hub_id")
    .distinct()
    .groupBy("hub_id")
    .agg(F.count("*").alias("total_package_hubs"))
)

hub_missing = (
    event_presence.groupBy("hub_id")
    .agg(
        F.sum(F.when(F.col("has_arrival") == 0, 1).otherwise(0)).alias("missing_arrival"),
        F.sum(F.when(F.col("has_departure") == 0, 1).otherwise(0)).alias("missing_departure"),
    )
)

hub_invalid_timestamp = (
    df_parsed.filter(F.col("ts").isNull())
    .groupBy("hub_id")
    .agg(F.count("*").alias("invalid_timestamp"))
)

hub_duplicates = (
    event_group_counts
    .withColumn("duplicate_extra", F.greatest(F.col("count") - 1, F.lit(0)))
    .groupBy("hub_id")
    .agg(F.sum("duplicate_extra").alias("duplicate_extra_scans"))
)

hub_invalid_order = (
    complete_pairs
    .filter(F.col("departure_ts") <= F.col("arrival_ts"))
    .groupBy("hub_id")
    .agg(F.count("*").alias("invalid_order_pairs"))
)

hub_valid_pairs = (
    valid_dwell.groupBy("hub_id")
    .agg(F.count("*").alias("valid_pairs"))
)

hub_quality = (
    hub_base
    .join(hub_missing, "hub_id", "left")
    .join(hub_invalid_timestamp, "hub_id", "left")
    .join(hub_duplicates, "hub_id", "left")
    .join(hub_invalid_order, "hub_id", "left")
    .join(hub_valid_pairs, "hub_id", "left")
    .fillna(0)
    .withColumn(
        "valid_pair_rate_pct",
        F.round(F.col("valid_pairs") / F.col("total_package_hubs") * 100, 2),
    )
    .orderBy("hub_id")
)

hub_quality.show(truncate=False)

+-------+------------------+---------------+-----------------+-----------------+---------------------+-------------------+-----------+-------------------+
|hub_id |total_package_hubs|missing_arrival|missing_departure|invalid_timestamp|duplicate_extra_scans|invalid_order_pairs|valid_pairs|valid_pair_rate_pct|
+-------+------------------+---------------+-----------------+-----------------+---------------------+-------------------+-----------+-------------------+
|HUB_BDG|247               |3              |19               |4                |14                   |2                  |223        |90.28              |
|HUB_BKS|88                |1              |16               |0                |3                    |1                  |70         |79.55              |
|HUB_DPS|198               |5              |15               |0                |7                    |1                  |177        |89.39              |
|HUB_JKT|879               |10             |59               |10      

## 6. Aggregate Dwell Time per Hub

In [35]:
hub_dwell_summary = (
    valid_dwell.groupBy("hub_id")
    .agg(
        F.count("*").alias("package_count"),
        F.round(F.avg("dwell_hours"), 2).alias("avg_dwell_hours"),
        F.round(F.percentile_approx("dwell_hours", 0.5, 10_000), 2).alias("median_dwell_hours"),
        F.round(F.percentile_approx("dwell_hours", 0.95, 10_000), 2).alias("p95_dwell_hours"),
        F.round(F.min("dwell_hours"), 2).alias("min_dwell_hours"),
        F.round(F.max("dwell_hours"), 2).alias("max_dwell_hours"),
        F.round(F.stddev("dwell_hours"), 2).alias("stddev_dwell_hours"),
    )
    .join(
        hub_quality.select(
            "hub_id",
            "total_package_hubs",
            "valid_pair_rate_pct",
            "missing_arrival",
            "missing_departure",
            "duplicate_extra_scans",
            "invalid_timestamp",
            "invalid_order_pairs",
        ),
        "hub_id",
        "left",
    )
    .orderBy(F.desc("avg_dwell_hours"))
    .cache()
)

hub_dwell_summary.show(truncate=False)

+-------+-------------+---------------+------------------+---------------+---------------+---------------+------------------+------------------+-------------------+---------------+-----------------+---------------------+-----------------+-------------------+
|hub_id |package_count|avg_dwell_hours|median_dwell_hours|p95_dwell_hours|min_dwell_hours|max_dwell_hours|stddev_dwell_hours|total_package_hubs|valid_pair_rate_pct|missing_arrival|missing_departure|duplicate_extra_scans|invalid_timestamp|invalid_order_pairs|
+-------+-------------+---------------+------------------+---------------+---------------+---------------+------------------+------------------+-------------------+---------------+-----------------+---------------------+-----------------+-------------------+
|HUB_MKS|539          |10.53          |10.61             |14.39          |2.91           |21.29          |2.44              |611               |88.22              |13             |52               |18                   |4  

## 7. Top 5 Bottleneck Hub dan Prioritas Investigasi

In [36]:
top_bottleneck_hubs = hub_dwell_summary.limit(5).cache()
top_bottleneck_hubs.show(truncate=False)

priority = top_bottleneck_hubs.first()
print("Prioritas investigasi:", priority["hub_id"])
print(
    f"avg={priority['avg_dwell_hours']:.2f} jam | "
    f"median={priority['median_dwell_hours']:.2f} jam | "
    f"p95={priority['p95_dwell_hours']:.2f} jam | "
    f"valid packages={priority['package_count']} | "
    f"valid-pair rate={priority['valid_pair_rate_pct']:.2f}%"
)

+-------+-------------+---------------+------------------+---------------+---------------+---------------+------------------+------------------+-------------------+---------------+-----------------+---------------------+-----------------+-------------------+
|hub_id |package_count|avg_dwell_hours|median_dwell_hours|p95_dwell_hours|min_dwell_hours|max_dwell_hours|stddev_dwell_hours|total_package_hubs|valid_pair_rate_pct|missing_arrival|missing_departure|duplicate_extra_scans|invalid_timestamp|invalid_order_pairs|
+-------+-------------+---------------+------------------+---------------+---------------+---------------+------------------+------------------+-------------------+---------------+-----------------+---------------------+-----------------+-------------------+
|HUB_MKS|539          |10.53          |10.61             |14.39          |2.91           |21.29          |2.44              |611               |88.22              |13             |52               |18                   |4  

## 8. Export Output

In [37]:
data_quality_rows = [
    ("total_events", int(total_events)),
    ("total_package_hubs", int(total_package_hubs)),
    ("missing_arrival", int(missing_arrival_count)),
    ("missing_departure", int(missing_departure_count)),
    ("invalid_timestamp", int(invalid_timestamp_count)),
    ("exact_duplicate_rows", int(exact_duplicate_count)),
    ("duplicate_event_groups", int(duplicate_event_groups)),
    ("duplicate_extra_scans", int(duplicate_extra_scans)),
    ("invalid_order_pairs", int(invalid_order_count)),
    ("valid_pairs", int(valid_pair_count)),
]
data_quality_summary = spark.createDataFrame(data_quality_rows, ["metric", "value"])

valid_dwell.orderBy("package_id", "hub_id").toPandas().to_csv(
    OUTPUT_DIR / "scan_pairs_dwell.csv", index=False
)
hub_dwell_summary.toPandas().to_csv(
    OUTPUT_DIR / "hub_dwell_summary.csv", index=False
)
top_bottleneck_hubs.toPandas().to_csv(
    OUTPUT_DIR / "top_bottleneck_hubs.csv", index=False
)
hub_quality.toPandas().to_csv(
    OUTPUT_DIR / "hub_quality_summary.csv", index=False
)
data_quality_summary.toPandas().to_csv(
    OUTPUT_DIR / "data_quality_summary.csv", index=False
)

print("Output berhasil disimpan:")
for output_file in sorted(OUTPUT_DIR.glob("*.csv")):
    print("-", output_file.name)

/home/rian/Documents/anteraja-capstone-rianyuliawan/bigdata-bottleneck-analysis/.venv/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/home/rian/Documents/anteraja-capstone-rianyuliawan/bigdata-bottleneck-analysis/.venv/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)
/home/rian/Documents/anteraja-capstone-rianyuliawan/bigdata-bottleneck-analysis/.venv/lib/python3.12/site-packages/pyspark/sql/pandas/types.py:778:

Output berhasil disimpan:
- data_quality_summary.csv
- hub_dwell_summary.csv
- hub_quality_summary.csv
- scan_pairs_dwell.csv
- top_bottleneck_hubs.csv


/home/rian/Documents/anteraja-capstone-rianyuliawan/bigdata-bottleneck-analysis/.venv/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/home/rian/Documents/anteraja-capstone-rianyuliawan/bigdata-bottleneck-analysis/.venv/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


## 9. Business Insight dan Rekomendasi

In [38]:
priority = top_bottleneck_hubs.first()

business_insight = (
    f"{priority['hub_id']} merupakan prioritas investigasi karena memiliki average dwell time tertinggi\n"
    f"sebesar {priority['avg_dwell_hours']:.2f} jam, median {priority['median_dwell_hours']:.2f} jam,\n"
    f"dan p95 {priority['p95_dwell_hours']:.2f} jam pada {priority['package_count']} paket valid.\n"
    f"Kualitas pasangannya mencakup valid-pair rate {priority['valid_pair_rate_pct']:.2f}%,\n"
    f"{priority['missing_arrival']} missing ARRIVAL, {priority['missing_departure']} missing DEPARTURE,\n"
    f"{priority['duplicate_extra_scans']} duplicate extra scan, {priority['invalid_timestamp']} timestamp tidak valid,\n"
    f"dan {priority['invalid_order_pairs']} pasangan dengan DEPARTURE sebelum ARRIVAL.\n"
    "Tim operasional sebaiknya memeriksa kapasitas sorting, antrean paket, jadwal keberangkatan line-haul,\n"
    "serta disiplin pemindaian pada hub tersebut, lalu memecah analisis per jam atau shift untuk menemukan\n"
    "periode penyebab dwell time tertinggi."
)

print(business_insight)

HUB_MKS merupakan prioritas investigasi karena memiliki average dwell time tertinggi
sebesar 10.53 jam, median 10.61 jam,
dan p95 14.39 jam pada 539 paket valid.
Kualitas pasangannya mencakup valid-pair rate 88.22%,
13 missing ARRIVAL, 52 missing DEPARTURE,
18 duplicate extra scan, 4 timestamp tidak valid,
dan 6 pasangan dengan DEPARTURE sebelum ARRIVAL.
Tim operasional sebaiknya memeriksa kapasitas sorting, antrean paket, jadwal keberangkatan line-haul,
serta disiplin pemindaian pada hub tersebut, lalu memecah analisis per jam atau shift untuk menemukan
periode penyebab dwell time tertinggi.
